## 1. Install Dependencies

In [6]:
!pip install -q beir ragas sentence-transformers langchain-community langchain-google-genai
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q accelerate

Looking in indexes: https://download.pytorch.org/whl/cu121


## 2. Imports & Configuration

In [7]:
import os
import glob
import time
import random
from typing import List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, adjusted_rand_score,
    accuracy_score, f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# ── Configuration ─────────────────────────────────────────────────────────────
CONFIG = {
    'model_name':      'bert-base-uncased',
    'latent_dim':      64,
    'curve_dims':      [256, 128, 64, 50, 32, 16],
    'epochs':          40,
    'batch_size':      512,
    'lr':              1e-3,
    'lambda_contrast': 2.5,
    'temperatures':    [0.04],
    'knn_pos':         20,
    'knn_centroid':    5,
    # ── Full-scale flags ───────────────────────────────────────────────────────
    # Set eval_subset to None to evaluate on the entire dataset.
    # For NDCG this means a 120k x 120k similarity matrix — use batch_size_eval
    # of 1024-2048 on A100 to keep it feasible.
    'eval_subset':         None,       # None = full dataset; int = subsample
    'batch_size_eval':     2048,       # larger batch for evaluation on A100
    'batch_size_encode':   1024,       # for SentenceTransformer encoding
    # uniformity uses pdist → O(N^2) — hard cap to avoid OOM
    'uniformity_max':      10000,
    'seed':                42,
}

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def setup_device():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')
        print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    return device

set_seed(CONFIG['seed'])
device = setup_device()

Using device: cuda


## 3. Data Loading & Embedding Functions

In [8]:
def load_data_split(split='train'):
    """Load sentence-compression dataset and return the text column."""
    print(f'Loading dataset (split={split})...')
    ds = load_dataset('embedding-data/sentence-compression', split=split)
    return ds[ds.column_names[0]]


def get_embeddings(sentences, model_name, device, batch_size=256):
    """Encode sentences with SentenceTransformer; returns CPU tensor."""
    print(f'Encoding {len(sentences)} sentences with {model_name}...')
    st = SentenceTransformer(model_name, device=device)
    return st.encode(
        sentences, convert_to_tensor=True,
        show_progress_bar=True,
        batch_size=batch_size,
    ).cpu()


def get_nearest_neighbors_gpu(embeddings, k, batch_size, device):
    """
    Exact k-NN via cosine similarity on GPU.
    Returns int64 numpy array (N, k).
    Column 0 = self (sim=1); use column 1 as positive pair.
    """
    print(f'Computing {k}-NN on GPU in batches (N={len(embeddings)})...')
    emb_gpu = F.normalize(embeddings.to(device), p=2, dim=1)
    parts = []
    for i in range(0, emb_gpu.size(0), batch_size):
        sim = torch.matmul(emb_gpu[i:i+batch_size], emb_gpu.T)
        parts.append(torch.topk(sim, k=k, dim=1).indices.cpu())
    del emb_gpu; torch.cuda.empty_cache()
    return torch.cat(parts, dim=0).numpy()

## 4. Generate / Load Base Embeddings  *(run once)*

In [9]:
EMBED_PATH = 'embeddings_bert.pt'

if os.path.exists(EMBED_PATH):
    print('Found existing embeddings_bert.pt — loading...')
    base_embeddings = torch.load(EMBED_PATH, map_location=device, weights_only=True)
else:
    print('Generating new BERT embeddings...')
    sentences = load_data_split(split='train')
    # sentences = sentences[:20000]  # Uncomment to limit size while debugging
    base_embeddings = get_embeddings(
        sentences, CONFIG['model_name'], device, batch_size=CONFIG['batch_size']
    )
    torch.save(base_embeddings, EMBED_PATH)
    print(f'Saved {EMBED_PATH} with shape {base_embeddings.shape}')

# Ensure float32 CPU tensor before training
if isinstance(base_embeddings, np.ndarray):
    base_embeddings = torch.from_numpy(base_embeddings).float()
else:
    base_embeddings = base_embeddings.float().cpu()

print(f'base_embeddings shape: {base_embeddings.shape}')

Generating new BERT embeddings...
Loading dataset (split=train)...
Encoding 180000 sentences with bert-base-uncased...


No sentence-transformers model found with name bert-base-uncased. Creating a new one with mean pooling.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/352 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 3.00 GiB. GPU 0 has a total capacity of 15.89 GiB of which 97.12 MiB is free. Process 3109 has 15.79 GiB memory in use. Of the allocated memory 14.75 GiB is allocated by PyTorch, and 764.94 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 5. Model & Loss Definitions

In [ ]:
class AutoEncoder(nn.Module):
    """768 -> 512 -> latent_dim -> 512 -> 768 autoencoder."""

    def __init__(self, input_dim=768, latent_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512), nn.LayerNorm(512), nn.ELU(),
            nn.Linear(512, 512),       nn.LayerNorm(512), nn.ELU(),
            nn.Linear(512, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 512), nn.LayerNorm(512), nn.ELU(),
            nn.Linear(512, 512),        nn.LayerNorm(512), nn.ELU(),
            nn.Linear(512, input_dim),
        )

    def forward(self, x):
        z = F.normalize(self.encoder(x), p=2, dim=1)  # L2-norm for cosine sim
        return z, self.decoder(z)


mse_loss_fn = nn.MSELoss()


def info_nce(z, z_pos, temperature):
    """Standard InfoNCE / NT-Xent contrastive loss."""
    z     = F.normalize(z,     dim=1)
    z_pos = F.normalize(z_pos, dim=1)
    logits = torch.matmul(z, z_pos.T) / temperature
    labels = torch.arange(z.size(0), device=z.device)
    return F.cross_entropy(logits, labels)


def tempspec_nce_loss(z, z_pos):
    actual_dim = z.shape[1]
    # Only use slice dims that fit within the actual latent dimension
    all_dims  = [16, 32, 50, 64, 128, 256]
    all_temps = [0.03, 0.05, 0.07, 0.09, 0.1, 0.1]
    loss = 0.0
    for d, t in zip(all_dims, all_temps):
        if d > actual_dim:
            break
        loss += info_nce(z[:, :d], z_pos[:, :d], t)
    return loss

## 6. Training Logic

In [ ]:
def train_one_temperature(tau, model, X, nn_indices, config, device):
    """
    Train AutoEncoder with TempSpecMRL + MSE reconstruction.

    Args:
        tau        : temperature (logged)
        model      : AutoEncoder already on device
        X          : base embeddings CPU float32 tensor, shape (N, 768)
        nn_indices : kNN index array (N, k) — numpy or GPU tensor
        config     : CONFIG dict
        device     : torch device
    """
    latent_dim = next(p for p in model.parameters() if p.dim() == 2).shape[0]
    # Detect actual latent dim from last encoder layer output shape
    latent_dim = list(model.encoder.parameters())[-1].shape[0]
    print(f'\n===== Training: tau={tau} | dim={latent_dim} =====')

    loader    = DataLoader(
        TensorDataset(X, torch.arange(len(X))),
        batch_size=config['batch_size'], shuffle=True,
        num_workers=4, pin_memory=True,     # faster data loading on cluster
    )
    optimizer = optim.Adam(model.parameters(), lr=config['lr'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'])

    # Move nn_indices to GPU tensor once — avoids CPU round-trip every batch
    if isinstance(nn_indices, np.ndarray):
        nn_indices_gpu = torch.tensor(nn_indices, dtype=torch.long).to(device)
    else:
        nn_indices_gpu = nn_indices.to(device)

    model.train()
    for epoch in range(config['epochs']):
        total_loss = total_rec = total_con = 0.0
        for x, idx in loader:
            x   = x.to(device, non_blocking=True)
            idx = idx.to(device, non_blocking=True)
            optimizer.zero_grad()

            z, x_hat = model(x)

            # Positive pair lookup stays on GPU — no .cpu() round-trip
            pos_idx  = nn_indices_gpu[idx, 1]
            z_pos, _ = model(X[pos_idx.cpu()].to(device, non_blocking=True))

            loss_rec = mse_loss_fn(x_hat, x)
            loss_nce = tempspec_nce_loss(z, z_pos)
            loss     = loss_rec + config['lambda_contrast'] * loss_nce

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            n = x.size(0)
            total_loss += loss.item()     * n
            total_rec  += loss_rec.item() * n
            total_con  += loss_nce.item() * n

        scheduler.step()
        if (epoch + 1) % 5 == 0:
            N = len(X)
            print(
                f'Epoch {epoch+1:>3}/{config["epochs"]} | '
                f'Loss: {total_loss/N:.4f} | Rec: {total_rec/N:.4f} | Con: {total_con/N:.4f}'
            )
    return model


def encode_latent(model, embeddings, batch_size, device):
    """Encode to latent space with L2 normalisation (mirrors model.forward)."""
    model.eval()
    if not isinstance(embeddings, torch.Tensor):
        embeddings = torch.tensor(embeddings, dtype=torch.float32)
    parts = []
    with torch.no_grad():
        for i in range(0, len(embeddings), batch_size):
            batch = embeddings[i:i+batch_size].to(device, non_blocking=True)
            z = F.normalize(model.encoder(batch), p=2, dim=1)
            parts.append(z.cpu())
    return torch.cat(parts, dim=0)

## 7. Run Training  *(Compression)*

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# COMPRESSION vs RETRIEVAL CURVE
# ══════════════════════════════════════════════════════════════════════════════

CURVE_DIMS   = CONFIG['curve_dims']
CURVE_METRIC = 'NDCG@10'
TAU          = CONFIG['temperatures'][0]

def encode_dim(latent_dim, embs_np, tau=TAU):
    path = f'model_compressed_bert_tau_{tau}_dim_{latent_dim}.pt'
    if not os.path.exists(path):
        print(f'  [SKIP] {path} not found.')
        return None
    ae = AutoEncoder(input_dim=768, latent_dim=latent_dim).to(device)
    ae.load_state_dict(torch.load(path, map_location=device, weights_only=True))
    ae.eval()
    parts = []
    t = torch.tensor(embs_np, dtype=torch.float32)
    with torch.no_grad():
        for i in range(0, len(embs_np), CONFIG['batch_size_eval']):
            z = ae.encoder(t[i:i+CONFIG['batch_size_eval']].to(device))
            parts.append(F.normalize(z, p=2, dim=1).cpu().numpy())
    return np.concatenate(parts, axis=0)


# ── Curve evaluation uses full corpus labels ──────────────────────────────────
# We can only build this cell after Cell 26 has run (all_labels_ret defined)
# base_sub_np is the FULL 768D embedding array — no subsampling
base_sub_np  = to_numpy(variations['BERT-Original (768D)'][0])
curve_labels = all_labels_ret   # full 120k labels

def eval_ndcg(embs_sub, labels, is_binary=False):
    m = compute_retrieval_metrics(
        q_embs=embs_sub, d_embs=embs_sub,
        q_labels=labels, d_labels=labels,
        k_values=[10], batch_size=CONFIG['batch_size_eval'],
        device=device, is_binary=is_binary,
    )
    return m['NDCG@10']


curve_points = []

# 1. 768D BERT baseline
print('Evaluating 768D baseline...')
curve_points.append({'label': 'BERT-768D', 'dim': 768,
                     'ndcg': eval_ndcg(base_sub_np, curve_labels), 'is_binary': False})

# 2. AE compressed dims
for dim in CURVE_DIMS:
    print(f'Evaluating AE dim={dim}...')
    embs = encode_dim(dim, base_sub_np)
    if embs is None:
        continue
    curve_points.append({'label': f'AE-{dim}D', 'dim': dim,
                         'ndcg': eval_ndcg(embs, curve_labels, is_binary=False), 'is_binary': False})
    bin_embs = binarize_embeddings(embs)
    curve_points.append({'label': f'AE-{dim}D-bin', 'dim': dim,
                         'ndcg': eval_ndcg(bin_embs, curve_labels, is_binary=True), 'is_binary': True})

# 3. PCA baselines
print('Evaluating PCA baselines...')
for dim in CURVE_DIMS:
    pca = PCA(n_components=dim).fit(base_sub_np)
    embs = pca.transform(base_sub_np)
    curve_points.append({'label': f'PCA-{dim}D', 'dim': dim,
                         'ndcg': eval_ndcg(embs, curve_labels, is_binary=False), 'is_binary': False})
    bin_embs = binarize_embeddings(embs)
    curve_points.append({'label': f'PCA-{dim}D-bin', 'dim': dim,
                         'ndcg': eval_ndcg(bin_embs, curve_labels, is_binary=True), 'is_binary': True})

# ── DataFrame & Plot ──────────────────────────────────────────────────────────
df_curve = pd.DataFrame(curve_points).sort_values('dim', ascending=False)
print('\n=== COMPRESSION vs RETRIEVAL TABLE (Full dataset) ===')
display(df_curve[['label', 'dim', 'ndcg', 'is_binary']].to_string(index=False))

fig, ax = plt.subplots(figsize=(13, 7))
ae_float  = df_curve[(df_curve['label'].str.startswith('AE'))  & (~df_curve['is_binary'])].sort_values('dim')
ae_bin    = df_curve[(df_curve['label'].str.startswith('AE'))  &  (df_curve['is_binary'])].sort_values('dim')
pca_float = df_curve[(df_curve['label'].str.startswith('PCA')) & (~df_curve['is_binary'])].sort_values('dim')
pca_bin   = df_curve[(df_curve['label'].str.startswith('PCA')) &  (df_curve['is_binary'])].sort_values('dim')
baseline  = df_curve[df_curve['label'] == 'BERT-768D']

C_BASELINE = '#212121'
C_AE_FLOAT = '#1565C0'
C_AE_BIN   = '#42A5F5'
C_PCA_FLOAT= '#C62828'
C_PCA_BIN  = '#EF9A9A'

ax.axhline(y=baseline['ndcg'].values[0], color=C_BASELINE, linestyle='--', linewidth=2,
           label=f'BERT-768D baseline ({baseline["ndcg"].values[0]:.4f})', zorder=5)
ax.plot(ae_float['dim'],  ae_float['ndcg'],  'o-', color=C_AE_FLOAT,  linewidth=2.5, markersize=9,
        markeredgecolor='white', markeredgewidth=1.5, label='AE Contrastive — float', zorder=4)
ax.plot(ae_bin['dim'],    ae_bin['ndcg'],    's--', color=C_AE_BIN,   linewidth=1.8, markersize=8,
        markeredgecolor=C_AE_FLOAT, markeredgewidth=1.2, label='AE Contrastive — binary', zorder=3)
ax.plot(pca_float['dim'], pca_float['ndcg'], '^-', color=C_PCA_FLOAT, linewidth=2.5, markersize=9,
        markeredgecolor='white', markeredgewidth=1.5, label='PCA — float', zorder=4)
ax.plot(pca_bin['dim'],   pca_bin['ndcg'],   'D--', color=C_PCA_BIN,  linewidth=1.8, markersize=8,
        markeredgecolor=C_PCA_FLOAT, markeredgewidth=1.2, label='PCA — binary', zorder=3)

if not ae_float.empty and not pca_float.empty:
    shared_dims = sorted(set(ae_float['dim']) & set(pca_float['dim']))
    ae_vals  = ae_float.set_index('dim').loc[shared_dims, 'ndcg'].values
    pca_vals = pca_float.set_index('dim').loc[shared_dims, 'ndcg'].values
    ax.fill_between(shared_dims, pca_vals, ae_vals, alpha=0.10, color=C_AE_FLOAT, label='AE advantage over PCA')

baseline_ndcg = baseline['ndcg'].values[0]
for _, row in ae_float.iterrows():
    pct = row['ndcg'] / baseline_ndcg * 100
    ax.annotate(f'{pct:.1f}%', xy=(row['dim'], row['ndcg']), xytext=(0, 10),
                textcoords='offset points', ha='center', fontsize=8.5, fontweight='bold', color=C_AE_FLOAT,
                arrowprops=dict(arrowstyle='-', color=C_AE_FLOAT, lw=0.8))
for _, row in pca_float.iterrows():
    pct = row['ndcg'] / baseline_ndcg * 100
    ax.annotate(f'{pct:.1f}%', xy=(row['dim'], row['ndcg']), xytext=(0, -16),
                textcoords='offset points', ha='center', fontsize=8.5, fontweight='bold', color=C_PCA_FLOAT,
                arrowprops=dict(arrowstyle='-', color=C_PCA_FLOAT, lw=0.8))

ax.set_xlabel('Embedding Dimension  (← more compressed)', fontsize=12)
ax.set_ylabel('NDCG@10', fontsize=12)
ax.set_title('Compression vs Retrieval Quality — Full AG News 120k\nAE Contrastive vs PCA  |  Float vs Binary',
             fontsize=13, fontweight='bold')
ax.set_xscale('log', base=2)
ax.set_xticks([16, 32, 50, 64, 128, 256, 768])
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.invert_xaxis()
ax.grid(True, alpha=0.25, linestyle='--')
ax.set_facecolor('#FAFAFA')
ax.legend(fontsize=10, ncol=2, loc='lower left', framealpha=0.9, edgecolor='#CCCCCC')
plt.tight_layout()
plt.savefig('compression_curve_full.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved compression_curve_full.png')

In [ ]:
print('Computing nearest neighbours for contrastive positive pairs...')
nn_indices = get_nearest_neighbors_gpu(
    base_embeddings, k=CONFIG['knn_pos'],
    batch_size=CONFIG['batch_size'], device=device,
)

for tau in CONFIG['temperatures']:
    print(f'\n>>> Compressing with tau={tau} <<<')

    for latent_dim in CONFIG['curve_dims']:
        print(f'\n--- Training dim={latent_dim} ---')
        model = AutoEncoder(input_dim=768, latent_dim=latent_dim).to(device)

        # No latent_dim argument — model is already built at the right dim
        model = train_one_temperature(tau, model, base_embeddings, nn_indices, CONFIG, device)

        save_name = f'model_compressed_bert_tau_{tau}_dim_{latent_dim}.pt'
        torch.save(model.state_dict(), save_name)
        print(f'Saved model -> {save_name}')

        z = encode_latent(model, base_embeddings, CONFIG['batch_size'], device)
        torch.save(z, f'embeddings_compressed_bert_tau_{tau}_dim_{latent_dim}.pt')
        print(f'Saved embeddings -> embeddings_compressed_bert_tau_{tau}_dim_{latent_dim}.pt')

## 8. Evaluation Setup — AG News Dataset

In [ ]:
def binarize_embeddings(embeddings):
    """
    Binarize embeddings to +/-1 using sign; zeros mapped to +1.
    Accepts both torch.Tensor and np.ndarray.
    """
    if isinstance(embeddings, torch.Tensor):
        binary = torch.sign(embeddings)
        return torch.where(binary == 0, torch.tensor(1.0, device=binary.device), binary)
    binary = np.sign(embeddings)
    return np.where(binary == 0, 1.0, binary)


def binary_similarity_matrix(query_bin, doc_bin):
    """Hamming-style dot product between sign vectors. Output shape: (Q, N)."""
    return np.dot(query_bin, doc_bin.T)


# ── Load AG News ──────────────────────────────────────────────────────────────
dataset    = load_dataset('ag_news')
train_data = dataset['train']

corpus = {}; queries = {}; qrels = {}
for idx, example in enumerate(train_data):
    sid = str(idx)
    corpus[sid]  = {'title': '', 'text': example['text']}
    queries[sid] = example['text']
    qrels[sid]   = {'label': example['label']}

docs_text = [corpus[did]['text'] for did in corpus]
print(f'Dataset Ready: {len(docs_text)} documents.')
print(f'Number of classes: {len(set(qrels[q]["label"] for q in qrels))}')

## 9. Evaluation Wrapper & Encoding

In [ ]:
class EvaluationWrapper:
    """Wraps SentenceTransformer with optional PCA and AutoEncoder compression."""

    def __init__(self, model_name, device):
        self.device = device
        self.pca_model = self.compressor = None
        print(f'Loading {model_name}...')
        self.base_model = SentenceTransformer(model_name, device=device)

    def encode_base(self, sentences, batch_size=None):
        bs = batch_size or CONFIG['batch_size_encode']
        return self.base_model.encode(
            sentences, convert_to_numpy=True,
            batch_size=bs, show_progress_bar=True,
        )

    def train_pca(self, embeddings, n_components=50):
        print(f'Training PCA to {n_components} dims on {len(embeddings)} samples...')
        self.pca_model = PCA(n_components=n_components).fit(embeddings)

    def encode_pca(self, embeddings):
        return self.pca_model.transform(embeddings)

    def load_compressor(self, model_path, input_dim=768, latent_dim=64):
        self.compressor = AutoEncoder(input_dim, latent_dim).to(self.device)
        self.compressor.load_state_dict(
            torch.load(model_path, map_location=self.device, weights_only=True)
        )
        self.compressor.eval()

    def encode_compressed(self, embeddings, batch_size=None):
        """Encode via AutoEncoder encoder with L2 normalisation."""
        bs = batch_size or CONFIG['batch_size_eval']
        t  = torch.tensor(embeddings, dtype=torch.float32)
        parts = []
        with torch.no_grad():
            for i in range(0, len(embeddings), bs):
                z = self.compressor.encoder(t[i:i+bs].to(self.device))
                parts.append(F.normalize(z, p=2, dim=1).cpu().numpy())
        return np.concatenate(parts, axis=0)


def to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.array(x)


# ── Initialise evaluators ─────────────────────────────────────────────────────
bert_eval   = EvaluationWrapper('bert-base-uncased',       device)
distil_eval = EvaluationWrapper('distilbert-base-uncased', device)

# Encode FULL corpus (120k) — uses batch_size_encode from CONFIG
print('Encoding full corpus with BERT-base...')
base_doc_embs   = bert_eval.encode_base(docs_text)
print('Encoding full corpus with DistilBERT...')
distil_doc_embs = distil_eval.encode_base(docs_text)

# Train PCA on full corpus embeddings
bert_eval.train_pca(base_doc_embs, n_components=64)

# Load best contrastive model
contrastive_ready = False
tau = CONFIG['temperatures'][0]
preferred_dim = CONFIG['latent_dim']
preferred_path = f'model_compressed_bert_tau_{tau}_dim_{preferred_dim}.pt'
model_files = glob.glob('model_compressed_bert_tau_*.pt')
if os.path.exists(preferred_path):
    bert_eval.load_compressor(preferred_path, latent_dim=preferred_dim)
    contrastive_ready = True
    print(f'Loaded compressor: {preferred_path}')
elif model_files:
    selected = model_files[0]
    bert_eval.load_compressor(selected)
    contrastive_ready = True
    print(f'Loaded compressor: {selected}')

# variations built over FULL corpus for all subsequent evaluation
# (no small query subset — queries == docs for AG News classification proxy)
variations = {
    'BERT-Original (768D)': (base_doc_embs,   base_doc_embs),
    'DistilBERT (768D)':    (distil_doc_embs, distil_doc_embs),
    'BERT-PCA (64D)':       (bert_eval.encode_pca(base_doc_embs),
                             bert_eval.encode_pca(base_doc_embs)),
}
if contrastive_ready:
    cq = bert_eval.encode_compressed(base_doc_embs)
    variations['BERT-Contrastive (64D)']           = (cq, cq)
    variations['BERT-Contrastive Binary (64-bit)'] = (binarize_embeddings(cq), binarize_embeddings(cq))

print(f'\nAll {len(variations)} variations ready.')
for name, (q, d) in variations.items():
    print(f'  {name}: shape={to_numpy(q).shape}')

## 10. Full-Dataset Re-encoding  *(AG News)*

In [ ]:
# Cell 18 already encodes the full corpus and builds variations.
# This cell is kept as a checkpoint — re-run if you need to reload
# embeddings from disk without re-encoding.

query_ids_all = list(queries.keys())
print(f'Total AG News samples: {len(query_ids_all)}')
print('variations keys:', list(variations.keys()))
for name, (q, d) in variations.items():
    print(f'  {name}: {to_numpy(q).shape}')

## 11. Embedding Geometry Metrics

In [ ]:
# ── Geometry caps ─────────────────────────────────────────────────────────────
# uniformity() uses torch.pdist → O(N^2) pairs.  Hard-cap to avoid OOM.
# All other metrics run on the full dataset.
UNIFORMITY_MAX = CONFIG['uniformity_max']   # 10 000
np.random.seed(CONFIG['seed'])


def uniformity(z, t=2):
    """Wang & Isola (2020) uniformity loss — lower is more uniform."""
    z = F.normalize(torch.tensor(to_numpy(z), dtype=torch.float32), dim=1)
    return torch.log(torch.mean(torch.exp(-t * torch.pdist(z, p=2).pow(2)))).item()


def iso_score(z):
    """Effective-dimensionality isotropy score in [0,1]; 1 = perfectly isotropic."""
    z = to_numpy(z); z = z - z.mean(0)
    ev = np.clip(np.linalg.eigvalsh(np.cov(z, rowvar=False)), 1e-12, None)
    d  = len(ev)
    return ((ev.sum()**2) / np.sum(ev**2) - 1) / (d - 1)


def linear_CKA(X, Y):
    """Centred Kernel Alignment between two embedding matrices."""
    X = to_numpy(X); X -= X.mean(0)
    Y = to_numpy(Y); Y -= Y.mean(0)
    return (np.linalg.norm(X.T @ Y, 'fro')**2 /
            (np.linalg.norm(X.T @ X, 'fro') * np.linalg.norm(Y.T @ Y, 'fro')))


def knn_overlap(X, Y, k=10, n_jobs=-1):
    """Mean fraction of k-NNs shared between X and Y spaces. Uses all CPU cores."""
    nX = NearestNeighbors(n_neighbors=k+1, n_jobs=n_jobs).fit(X).kneighbors(return_distance=False)[:, 1:]
    nY = NearestNeighbors(n_neighbors=k+1, n_jobs=n_jobs).fit(Y).kneighbors(return_distance=False)[:, 1:]
    return float(np.mean([len(set(nX[i]) & set(nY[i])) / k for i in range(len(X))]))


base_full = to_numpy(variations['BERT-Original (768D)'][0])
N         = base_full.shape[0]
print(f'Full dataset size for geometry: {N}')

# Subsample index only for uniformity
uni_idx  = np.random.choice(N, min(N, UNIFORMITY_MAX), replace=False)
base_uni = base_full[uni_idx]

geo_results = {}
for name, (embs, _) in variations.items():
    embs_np = to_numpy(embs)
    print(f'Geometry [{name}] on {N} samples...')
    geo_results[name] = {
        # uniformity: capped at UNIFORMITY_MAX to avoid OOM
        'Uniformity':      uniformity(embs_np[uni_idx]),
        # all others: full dataset
        'IsoScore':        iso_score(embs_np),
        'CKA vs Original': linear_CKA(base_full, embs_np),
        'KNN@10 Overlap':  knn_overlap(base_full, embs_np, k=10),
    }

print('\n=== EMBEDDING GEOMETRY (Full dataset, except Uniformity capped at 10k) ===')
display(pd.DataFrame(geo_results).T)

## 12. Classification Evaluation

In [ ]:
# Full dataset classification — all 120k samples
np.random.seed(CONFIG['seed'])

all_labels = np.array([qrels[qid]['label'] for qid in query_ids_all])
N_cls      = len(all_labels)
cls_idx    = np.arange(N_cls)   # use ALL samples
labels     = all_labels[cls_idx]
print(f'Classification on full dataset: {N_cls} samples')

results_cls = {}
for name, (embs, _) in variations.items():
    print(f'\nClassification [{name}]')
    X = to_numpy(embs)[cls_idx]; y = labels

    # KMeans with silhouette approximation — full 120k is too slow for exact silhouette
    km   = KMeans(n_clusters=len(np.unique(y)), random_state=42, n_init=5, max_iter=100)
    pred = km.fit_predict(X)
    sil  = silhouette_score(X, pred, sample_size=5000, random_state=42)  # approximated
    ari  = adjusted_rand_score(y, pred)

    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # n_jobs=-1 uses all available CPU cores
    clf = LogisticRegression(max_iter=1000, n_jobs=-1)
    clf.fit(Xtr, ytr); p = clf.predict(Xte)
    log_acc = accuracy_score(yte, p); log_f1 = f1_score(yte, p, average='macro')

    clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    clf.fit(Xtr, ytr); p = clf.predict(Xte)
    rf_acc = accuracy_score(yte, p); rf_f1 = f1_score(yte, p, average='macro')

    clf = LinearSVC(max_iter=2000, random_state=42)
    clf.fit(Xtr, ytr); p = clf.predict(Xte)
    svm_acc = accuracy_score(yte, p); svm_f1 = f1_score(yte, p, average='macro')

    results_cls[name] = {
        'Silhouette (approx)': sil, 'ARI': ari,
        'LogReg Acc': log_acc, 'LogReg F1': log_f1,
        'RF Acc': rf_acc,      'RF F1': rf_f1,
        'LinearSVM Acc': svm_acc, 'LinearSVM F1': svm_f1,
    }

print('\n=== CLASSIFICATION EVALUATION (Full dataset) ===')
display(pd.DataFrame(results_cls).T)

## 13. Retrieval Metrics — Recall, MRR, NDCG

> **Note:** AG News has no ground-truth qrels, so we simulate retrieval relevance using class labels.  
> A retrieved document is *relevant* if it shares the **same class label** as the query.  
> This is a standard proxy for evaluating embedding quality on classification datasets.

In [ ]:
# ── Metric helpers ────────────────────────────────────────────────────────────

def recall_at_k(retrieved_labels, query_label, k):
    return float(np.sum(retrieved_labels[:k] == query_label)) / k


def mrr_at_k(retrieved_labels, query_label, k):
    for rank, label in enumerate(retrieved_labels[:k], start=1):
        if label == query_label:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(retrieved_labels, query_label, k):
    gains     = (retrieved_labels[:k] == query_label).astype(float)
    discounts = np.log2(np.arange(2, k + 2))
    dcg       = float(np.sum(gains / discounts))
    idcg      = float(np.sum(np.ones(k) / discounts))
    return dcg / idcg if idcg > 0 else 0.0


def compute_retrieval_metrics(
    q_embs, d_embs, q_labels, d_labels,
    k_values=(1, 5, 10), batch_size=2048,
    device=torch.device('cpu'), is_binary=False,
):
    """
    Compute Recall@k, MRR@k, NDCG@k.
    batch_size controls the (B x N) score matrix size per step.
    On A100 80GB, batch_size=2048 against 120k docs uses ~3.7GB VRAM per step.
    """
    max_k = max(k_values)
    n_q   = q_embs.shape[0]
    q_t   = torch.tensor(q_embs, dtype=torch.float32).to(device)
    d_t   = torch.tensor(d_embs, dtype=torch.float32).to(device)

    if not is_binary:
        q_t = F.normalize(q_t, p=2, dim=1)
        d_t = F.normalize(d_t, p=2, dim=1)

    acc = {f'{m}@{k}': 0.0 for m in ['Recall', 'MRR', 'NDCG'] for k in k_values}

    for start in range(0, n_q, batch_size):
        end    = min(start + batch_size, n_q)
        scores = torch.matmul(q_t[start:end], d_t.T)   # (B, N)

        # Mask self-matches
        for local_i in range(end - start):
            scores[local_i, start + local_i] = -1e9

        _, top_inds = torch.topk(scores, k=max_k, dim=1)
        top_np      = top_inds.cpu().numpy()

        for local_i, global_i in enumerate(range(start, end)):
            q_lbl    = q_labels[global_i]
            ret_lbls = d_labels[top_np[local_i]]
            for k in k_values:
                acc[f'Recall@{k}'] += recall_at_k(ret_lbls, q_lbl, k)
                acc[f'MRR@{k}']    += mrr_at_k(ret_lbls,    q_lbl, k)
                acc[f'NDCG@{k}']   += ndcg_at_k(ret_lbls,   q_lbl, k)

    return {key: val / n_q for key, val in acc.items()}


# ── Run retrieval evaluation on FULL dataset ──────────────────────────────────
K_VALUES = [1, 5, 10]

np.random.seed(CONFIG['seed'])
all_labels_ret = np.array([qrels[qid]['label'] for qid in query_ids_all])
N_ret          = len(all_labels_ret)

# eval_subset=None → full dataset; set to an int (e.g. 50000) to subsample
EVAL_SUBSET = CONFIG['eval_subset']
if EVAL_SUBSET is not None and N_ret > EVAL_SUBSET:
    ret_idx    = np.random.choice(N_ret, EVAL_SUBSET, replace=False)
    ret_labels = all_labels_ret[ret_idx]
    print(f'Evaluating on {EVAL_SUBSET} subsampled queries (eval_subset set in CONFIG)')
else:
    ret_idx    = np.arange(N_ret)
    ret_labels = all_labels_ret
    print(f'Evaluating on FULL dataset: {N_ret} queries')

retrieval_results = {}
for name, (embs, _) in variations.items():
    print(f'\nRetrieval metrics [{name}] on {len(ret_idx)} queries...', flush=True)
    t0       = time.time()
    embs_sub = to_numpy(embs)[ret_idx]
    metrics  = compute_retrieval_metrics(
        q_embs    = embs_sub,
        d_embs    = embs_sub,
        q_labels  = ret_labels,
        d_labels  = ret_labels,
        k_values  = K_VALUES,
        batch_size = CONFIG['batch_size_eval'],   # 2048 for A100
        device    = device,
        is_binary = 'Binary' in name,
    )
    retrieval_results[name] = metrics
    elapsed = time.time() - t0
    print(f'  done in {elapsed:.1f}s ({elapsed/60:.1f} min)')

col_order = [f'{m}@{k}' for m in ['Recall', 'MRR', 'NDCG'] for k in K_VALUES]
df_ret    = pd.DataFrame(retrieval_results).T[col_order]

print('\n=== RETRIEVAL METRICS (Full dataset) ===')
print('Higher is better for all metrics.\n')
display(df_ret.style.format('{:.4f}').highlight_max(axis=0, color='lightgreen'))

## 14. Asymmetric Re-ranking  *(Binary → Float)*

In [ ]:
def evaluate_asymmetric_reranking(
    q_embs_float, d_embs_float, q_embs_bin, d_embs_bin,
    top_k_rerank=100, final_k=10,
):
    """
    Two-stage retrieval:
      Stage 1 — binary Hamming dot-product to retrieve top_k_rerank candidates.
      Stage 2 — exact float cosine re-ranking to final_k.
    Returns global document indices, shape (Q, final_k).
    """
    print(f'Asymmetric Re-ranking: Binary top-{top_k_rerank} -> Float top-{final_k}')
    # Ensure float32 before converting to tensor (numpy defaults to float64)
    d_flt = F.normalize(torch.tensor(d_embs_float.astype(np.float32)).to(device), p=2, dim=1)
    q_flt = F.normalize(torch.tensor(q_embs_float.astype(np.float32)).to(device), p=2, dim=1)
    q_bin = torch.tensor(q_embs_bin.astype(np.float32)).to(device)
    d_bin = torch.tensor(d_embs_bin.astype(np.float32)).to(device)

    all_inds = []; t0 = time.time()
    for start in range(0, q_bin.shape[0], 512):
        end = min(start + 512, q_bin.shape[0])  # clamp to avoid empty slice
        qb  = q_bin[start:end]; qf = q_flt[start:end]
        _, top100 = torch.topk(torch.matmul(qb, d_bin.T), k=top_k_rerank, dim=1)
        batch_inds = []
        for i in range(qf.size(0)):
            cand  = d_flt[top100[i]]
            exact = torch.matmul(qf[i].unsqueeze(0), cand.T).squeeze(0)
            _, bi = torch.topk(exact, k=final_k)
            batch_inds.append(top100[i][bi].unsqueeze(0))
        all_inds.append(torch.cat(batch_inds, dim=0))

    out = torch.cat(all_inds, dim=0).cpu().numpy()
    print(f'Completed in {time.time()-t0:.2f}s | output shape: {out.shape}')
    return out


if 'BERT-Original (768D)' in variations and 'BERT-Contrastive (50D)' in variations:
    bqf, bdf = variations['BERT-Original (768D)']
    cqf, cdf = variations['BERT-Contrastive (50D)']
    # to_numpy ensures we always pass np.ndarray regardless of tensor/array state
    final_asym_inds = evaluate_asymmetric_reranking(
        q_embs_float=to_numpy(bqf), d_embs_float=to_numpy(bdf),
        q_embs_bin=to_numpy(binarize_embeddings(cqf)),
        d_embs_bin=to_numpy(binarize_embeddings(cdf)),
        top_k_rerank=100, final_k=10,
    )
    print('Shape of final retrieved indices:', final_asym_inds.shape)
else:
    print('Variations required for asymmetric testing are missing.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY ANALYSIS: Performance Retention, Binarization Retention, Re-ranking Gain
# Depends on: retrieval_results (Cell 13), final_asym_inds (Cell 14), ret_labels
# ══════════════════════════════════════════════════════════════════════════════

BASELINE_KEY     = 'BERT-Original (768D)'
COMPRESSED_KEY   = 'BERT-Contrastive (50D)'
BINARY_KEY       = 'BERT-Contrastive Binary (50-bit)'
SUMMARY_METRIC   = 'NDCG@10'   # primary metric used for all retention calculations

assert BASELINE_KEY   in retrieval_results, f"Missing: {BASELINE_KEY}"
assert COMPRESSED_KEY in retrieval_results, f"Missing: {COMPRESSED_KEY} — run training first"

# ── 1. Performance Retention (%) ──────────────────────────────────────────────
# How much retrieval quality does each 50D model retain vs the 768D baseline?
print("=" * 65)
print("1. PERFORMANCE RETENTION  (50D vs 768D baseline)")
print("=" * 65)

baseline_scores = retrieval_results[BASELINE_KEY]
retention_rows  = {}

for name, metrics in retrieval_results.items():
    if name == BASELINE_KEY:
        continue
    row = {}
    for metric, baseline_val in baseline_scores.items():
        model_val      = metrics[metric]
        retention_pct  = (model_val / baseline_val * 100) if baseline_val > 0 else 0.0
        row[metric]    = retention_pct
    retention_rows[name] = row

df_retention = pd.DataFrame(retention_rows).T[col_order]
display(df_retention.style
        .format('{:.2f}%')
        .highlight_max(axis=0, color='lightgreen')
        .highlight_min(axis=0, color='#ffcccc'))

# Plain-english summary for the primary metric
print(f"\nSummary ({SUMMARY_METRIC}):")
for name, row in retention_rows.items():
    val = row[SUMMARY_METRIC]
    print(f"  {name:<40} retains {val:.1f}% of baseline quality")


# ── 2. Binarization Retention ─────────────────────────────────────────────────
# How much quality is LOST when going from 50D float → 50-bit binary?
# Following the paper: contrastive models should retain far more than PCA/original.
print("\n" + "=" * 65)
print("2. BINARIZATION RETENTION  (50D float → 50-bit binary)")
print("=" * 65)

float_sources = {
    'BERT-PCA (50D)':            'BERT-PCA Binary' if 'BERT-PCA Binary' in retrieval_results
                                  else None,
    COMPRESSED_KEY:              BINARY_KEY if BINARY_KEY in retrieval_results else None,
}

bin_rows = {}
for float_key, binary_key in float_sources.items():
    if binary_key is None or binary_key not in retrieval_results:
        # PCA binary wasn't computed — skip gracefully
        continue
    float_scores  = retrieval_results[float_key]
    binary_scores = retrieval_results[binary_key]
    row = {}
    for metric in col_order:
        fval          = float_scores[metric]
        bval          = binary_scores[metric]
        retained_pct  = (bval / fval * 100) if fval > 0 else 0.0
        loss_pct      = 100.0 - retained_pct
        row[metric]   = retained_pct
    bin_rows[f'{float_key} -> Binary'] = row

if bin_rows:
    df_bin = pd.DataFrame(bin_rows).T[col_order]
    display(df_bin.style
            .format('{:.2f}%')
            .highlight_max(axis=0, color='lightgreen')
            .highlight_min(axis=0, color='#ffcccc'))

    print(f"\nSummary ({SUMMARY_METRIC}):")
    for label, row in bin_rows.items():
        val      = row[SUMMARY_METRIC]
        loss     = 100.0 - val
        print(f"  {label:<50} retains {val:.1f}%  (loss: {loss:.1f}%)")
    print("\n  Higher retention = less quality lost during binarization.")
    print("  The paper predicts contrastive > PCA/standard on this metric.")
else:
    print("  Binary variations not found in retrieval_results.")
    print("  Make sure 'BERT-Contrastive Binary (50-bit)' ran in Cell 13.")


# ── 3. Re-ranking Gain ────────────────────────────────────────────────────────
# Compare: 50D contrastive retrieval alone  vs
#          50D binary first-stage + 768D BERT re-rank (final_asym_inds)
# Uses final_asym_inds from Cell 14 and ret_labels / ret_idx from Cell 13.
# ── 3. Re-ranking Gain ────────────────────────────────────────────────────────
# ── 3. Re-ranking Gain ────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("3. RE-RANKING GAIN  (50D retrieval vs Binary→768D re-rank)")
print("=" * 65)

if 'final_asym_inds' not in locals():
    print("  final_asym_inds not found — run Cell 14 (Asymmetric Re-ranking) first.")
else:
    final_k = final_asym_inds.shape[1]   # = 10

    # Cap to the subsampled eval set used in Cell 13
    # final_asym_inds may cover the full corpus but ret_labels is only 10k
    n_q = min(final_asym_inds.shape[0], len(ret_labels))
    eval_labels_rerank = ret_labels[:n_q]           # query labels (subsampled)
    asym_inds_sub      = final_asym_inds[:n_q]      # only take matching rows

    rerank_acc = {f'{m}@{k}': 0.0
                  for m in ['Recall', 'MRR', 'NDCG']
                  for k in K_VALUES if k <= final_k}

    for qi in range(n_q):
        q_lbl    = eval_labels_rerank[qi]
        ret_lbls = all_labels_ret[asym_inds_sub[qi]]   # global indices → full label array
        for k in K_VALUES:
            if k > final_k:
                continue
            rerank_acc[f'Recall@{k}'] += recall_at_k(ret_lbls, q_lbl, k)
            rerank_acc[f'MRR@{k}']    += mrr_at_k(ret_lbls,    q_lbl, k)
            rerank_acc[f'NDCG@{k}']   += ndcg_at_k(ret_lbls,   q_lbl, k)

    rerank_metrics = {k: v / n_q for k, v in rerank_acc.items()}

    comparison = {
        '768D BERT (baseline)':       {m: retrieval_results[BASELINE_KEY][m]   for m in rerank_metrics},
        '50D Contrastive only':       {m: retrieval_results[COMPRESSED_KEY][m] for m in rerank_metrics},
        'Binary→768D Re-rank (ours)': rerank_metrics,
    }

    df_rerank = pd.DataFrame(comparison).T
    rerank_col_order = [c for c in col_order if c in df_rerank.columns]
    display(df_rerank[rerank_col_order].style
            .format('{:.4f}')
            .highlight_max(axis=0, color='lightgreen'))

    base_50d = retrieval_results[COMPRESSED_KEY][SUMMARY_METRIC]
    rerank_v = rerank_metrics[SUMMARY_METRIC]
    gain_abs = rerank_v - base_50d
    gain_pct = gain_abs / base_50d * 100 if base_50d > 0 else 0.0

    print(f"\nGain of re-ranking over 50D-only ({SUMMARY_METRIC}):")
    print(f"  50D Contrastive alone : {base_50d:.4f}")
    print(f"  Binary→768D Re-rank   : {rerank_v:.4f}")
    print(f"  Absolute gain         : {gain_abs:+.4f}")
    print(f"  Relative gain         : {gain_pct:+.2f}%")

    gap = rerank_v - retrieval_results[BASELINE_KEY][SUMMARY_METRIC]
    print(f"\n  Gap vs 768D baseline  : {gap:+.4f}  "
          f"({'above' if gap >= 0 else 'below'} baseline)")